In [1]:
import warnings
warnings.filterwarnings("ignore")

import polars as pl
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
df = pl.read_parquet(
    "../data/processed/merged_data.parquet"
)

print(df.shape)

(590540, 434)


In [3]:
missing_df = (
    df.null_count()
    .transpose(include_header=True)
)

missing_df.columns = ["column", "missing_count"]

missing_df = missing_df.with_columns(
    (
        pl.col("missing_count") / len(df) * 100
    ).alias("missing_percentage")
)

drop_cols = (
    missing_df
    .filter(pl.col("missing_percentage") > 90)
    ["column"]
    .to_list()
)

print("Columns to drop:", len(drop_cols))

Columns to drop: 12


In [4]:
df = df.drop(drop_cols)

print(df.shape)

(590540, 422)


In [5]:
df = df.to_pandas()

In [6]:
categorical_cols = df.select_dtypes(
    include=["object"]
).columns

print("Categorical columns:", len(categorical_cols))

Categorical columns: 29


In [7]:
for col in categorical_cols:
    
    df[col] = df[col].astype(str)

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

In [9]:
imputer = SimpleImputer(strategy="median")

df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

df = df_imputed

In [10]:
X = df.drop("isFraud", axis=1)

y = df["isFraud"]

print(X.shape)
print(y.shape)

(590540, 421)
(590540,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(472432, 421)
(118108, 421)


In [12]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
rf_probs = rf_model.predict_proba(X_test)[:, 1]

rf_preds = rf_model.predict(X_test)

In [14]:
rf_roc_auc = roc_auc_score(y_test, rf_probs)

rf_pr_auc = average_precision_score(y_test, rf_probs)

print("Random Forest ROC AUC:", rf_roc_auc)

print("Random Forest PR AUC:", rf_pr_auc)

Random Forest ROC AUC: 0.8685815562148389
Random Forest PR AUC: 0.5192626774057562


In [15]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=64,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.748247 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37872
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 419
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,10
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
lgbm_probs = lgbm_model.predict_proba(X_test)[:, 1]

lgbm_roc_auc = roc_auc_score(y_test, lgbm_probs)

lgbm_pr_auc = average_precision_score(
    y_test,
    lgbm_probs
)

print("LightGBM ROC AUC:", lgbm_roc_auc)

print("LightGBM PR AUC:", lgbm_pr_auc)

LightGBM ROC AUC: 0.9488005958068813
LightGBM PR AUC: 0.7569270707902008


In [17]:
cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

cat_model.fit(X_train, y_train)

CatBoostClassifier(depth=8, iterations=300, learning_rate=0.05, verbose=0)

In [18]:
cat_probs = cat_model.predict_proba(X_test)[:, 1]

cat_roc_auc = roc_auc_score(y_test, cat_probs)

cat_pr_auc = average_precision_score(
    y_test,
    cat_probs
)

print("CatBoost ROC AUC:", cat_roc_auc)

print("CatBoost PR AUC:", cat_pr_auc)

CatBoost ROC AUC: 0.9073197739113923
CatBoost PR AUC: 0.6370754136051348


In [19]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [20]:
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

xgb_roc_auc = roc_auc_score(y_test, xgb_probs)

xgb_pr_auc = average_precision_score(
    y_test,
    xgb_probs
)

print("XGBoost ROC AUC:", xgb_roc_auc)

print("XGBoost PR AUC:", xgb_pr_auc)

XGBoost ROC AUC: 0.9500562015973913
XGBoost PR AUC: 0.7566580319064711


In [21]:
results = pd.DataFrame({
    "Model": [
        "RandomForest",
        "LightGBM",
        "CatBoost",
        "XGBoost"
    ],
    "ROC_AUC": [
        rf_roc_auc,
        lgbm_roc_auc,
        cat_roc_auc,
        xgb_roc_auc
    ],
    "PR_AUC": [
        rf_pr_auc,
        lgbm_pr_auc,
        cat_pr_auc,
        xgb_pr_auc
    ]
})

results.sort_values(
    "ROC_AUC",
    ascending=False
)

,Model,ROC_AUC,PR_AUC
3,XGBoost,0.950056,0.756658
1,LightGBM,0.948801,0.756927
2,CatBoost,0.907320,0.637075
0,RandomForest,0.868582,0.519263
